# HI 5312 | MIMIC-IV Demo: pandas Joins & Clinical Analytics
**Hands-on student workbook** | GitHub Codespaces / Jupyter | No PostgreSQL required

**Learning outcomes:** load compressed CSVs; identify patient, admission and event-level keys; perform safe joins; aggregate before joining; explore diagnoses, labs and ICU utilization; create reproducible project analyses.

**Important:** MIMIC-IV Demo is a small, selected, deidentified teaching sample. Its results are not population estimates or clinical recommendations. Timestamps are deidentified. Work only with approved course data.

## 0. How to use this notebook
Run cells **top to bottom** (Jupyter: *Run All*). A `NameError` often means a preceding setup cell did not run. The setup supports both `.csv` and `.csv.gz` MIMIC-IV Demo files, and searches the usual course Codespace locations. No PostgreSQL connection is required for this pandas notebook.

Most examples use the `hosp` files: `patients`, `admissions`, `diagnoses_icd`, `d_icd_diagnoses`, `labevents`, `d_labitems`, `prescriptions`, `transfers`. ICU examples use `icustays` when available. Some tables are optional; the notebook checks before loading them.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print("pandas:", pd.__version__)
print("Current directory:", Path.cwd())

## 1. Locate and load MIMIC-IV Demo CSVs
The course files may be `.csv` **or** `.csv.gz`. pandas reads either format directly. No database credentials needed. Run both setup cells before continuing.


In [ ]:
# Prefer the official course Codespace path; search other workspace locations if needed.
roots = [Path("/workspaces"), Path.cwd()]
known_folders = [
    Path("/workspaces/hi5304-notebooks/data/mimic-iv-demo/hosp"),
    Path("/workspaces/hi5304-notebook/data/mimic-iv-demo/hosp"),
]

def has_patients(folder):
    return any((folder / filename).is_file() for filename in ("patients.csv", "patients.csv.gz"))

candidates = {p.resolve() for p in known_folders if has_patients(p)}
if not candidates:
    for root in roots:
        if root.exists():
            for pattern in ("**/hosp/patients.csv", "**/hosp/patients.csv.gz"):
                candidates.update(p.resolve().parent for p in root.glob(pattern))

candidates = sorted(candidates, key=lambda p: ("hi5304-notebooks" not in p.parts, str(p)))
print("MIMIC hospital folders found:")
for i, folder in enumerate(candidates, 1):
    print(f"  {i}: {folder}")
if not candidates:
    raise FileNotFoundError(
        "Could not find hosp/patients.csv or hosp/patients.csv.gz. "
        "Upload the MIMIC-IV Demo hosp and icu folders under "
        "/workspaces/hi5304-notebooks/data/mimic-iv-demo/."
    )
SELECTED_HOSP_FOLDER = candidates[0]
print("\nSelected:", SELECTED_HOSP_FOLDER)
# If needed, override with: SELECTED_HOSP_FOLDER = Path("/your/path/to/hosp")


In [ ]:
hosp = Path(SELECTED_HOSP_FOLDER)
icu = hosp.parent / "icu"

def load_table(name, folder=None, required=True):
    if folder is None:
        folder = hosp
    folder = Path(folder)
    paths = [folder / f"{name}.csv", folder / f"{name}.csv.gz"]
    path = next((p for p in paths if p.is_file()), None)
    if path is None:
        if required:
            raise FileNotFoundError(
                f"Missing required MIMIC table '{name}' in {folder}. "
                "Expected a .csv or .csv.gz file."
            )
        print(f"Optional table not found: {name} ({folder})")
        return None
    result = pd.read_csv(path, low_memory=False)
    print(f"{name}: {result.shape[0]:,} rows x {result.shape[1]} columns ({path.name})")
    return result

patients = load_table("patients")
admissions = load_table("admissions")
print("\nPatient columns:", patients.columns.tolist())
print("Admission columns:", admissions.columns.tolist())
display(patients.head(3), admissions.head(3))


## 2. Start with the data grain and join keys
- `subject_id`: one patient; one patient may have multiple hospitalizations.
- `hadm_id`: one hospital admission; an admission may have multiple diagnoses, lab events, transfers, and ICU stays.
- `stay_id`: one ICU stay (when using ICU tables).
- `itemid`: an event/test code; use a dictionary to interpret it.

**Rule:** Decide whether your result should have one row per *patient*, *admission*, *ICU stay*, or *event* **before** merging.

In [ ]:
print("Unique patients:", patients.subject_id.nunique())
print("Unique admissions:", admissions.hadm_id.nunique())
print("Admissions per patient (summary):")
display(admissions.groupby("subject_id").hadm_id.nunique().describe())
print("Patient key duplicates:", patients.subject_id.duplicated().sum())
print("Admission key duplicates:", admissions.hadm_id.duplicated().sum())

## 3. JOIN #1 — Admissions + patients (many-to-one, LEFT)
Keep every admission, adding patient characteristics. `validate='many_to_one'` catches unexpected duplicate patient keys. `indicator=True` helps audit unmatched records.

In [ ]:
patient_admissions = admissions.merge(
    patients, on="subject_id", how="left", validate="many_to_one", indicator=True
)
print("Before:", len(admissions), "After:", len(patient_admissions))
print(patient_admissions["_merge"].value_counts())
assert len(patient_admissions) == len(admissions)
display(patient_admissions[["subject_id", "hadm_id", "anchor_age", "gender", "admission_type"]].head())

## 4. JOIN #2 — INNER vs LEFT vs ANTI join
An inner join keeps matched admissions only; a left join keeps all admissions. An anti join finds admission rows without a patient match. These are useful data-quality checks.

In [ ]:
inner = admissions.merge(patients[["subject_id", "gender"]], on="subject_id", how="inner", validate="many_to_one")
left = admissions.merge(patients[["subject_id", "gender"]], on="subject_id", how="left", validate="many_to_one", indicator=True)
unmatched = left.loc[left["_merge"] == "left_only"]
print("Inner rows:", len(inner), "Left rows:", len(left), "Unmatched admissions:", len(unmatched))

## 5. Admissions analytics — time, length of stay, and readmissions
Calculate length of stay from admission and discharge timestamps. Use the admission-level table, not a lab-event-expanded table, to avoid double counting.

In [ ]:
admissions_a = admissions.copy()
for c in ["admittime", "dischtime"]:
    admissions_a[c] = pd.to_datetime(admissions_a[c], errors="coerce")
admissions_a["los_days"] = (admissions_a.dischtime - admissions_a.admittime).dt.total_seconds() / 86400
print(admissions_a.los_days.describe())
display(admissions_a.groupby("admission_type", dropna=False).los_days.agg(["count", "mean", "median"]).round(2).sort_values("count", ascending=False).head(12))

In [ ]:
# Number of admissions per patient; note: this is not a validated 30-day readmission measure.
admissions_per_patient = admissions_a.groupby("subject_id").agg(
    n_admissions=("hadm_id", "nunique"),
    mean_los_days=("los_days", "mean")
).reset_index()
display(admissions_per_patient.head())
print("Patients with >1 admissions:", (admissions_per_patient.n_admissions > 1).sum())
admissions_per_patient.n_admissions.value_counts().sort_index().plot(kind="bar", title="Admissions per patient")
plt.xlabel("Number of admissions"); plt.ylabel("Patients"); plt.show()

## 6. JOIN #3 — Patient-level aggregation + patients
First aggregate admissions to one row per `subject_id`, then merge. This preserves one row per patient and is often ideal for student project datasets.

In [ ]:
patient_summary = patients.merge(admissions_per_patient, on="subject_id", how="left", validate="one_to_one")
patient_summary["n_admissions"] = patient_summary.n_admissions.fillna(0).astype(int)
assert len(patient_summary) == len(patients)
display(patient_summary[["subject_id", "anchor_age", "gender", "n_admissions", "mean_los_days"]].head())
display(patient_summary.groupby("gender", dropna=False).agg(
    patients=("subject_id", "nunique"), mean_anchor_age=("anchor_age", "mean"), mean_admissions=("n_admissions", "mean")
).round(2))

## 7. JOIN #4 — Diagnoses + ICD code dictionary (many-to-one)
ICD descriptions require **both** `icd_code` and `icd_version`; joining on the code alone can produce incorrect matches or duplicate rows. `seq_num` represents diagnosis ordering, not severity.

In [ ]:
diagnoses = load_table("diagnoses_icd")
icd_dictionary = load_table("d_icd_diagnoses")
print("Diagnosis columns:", diagnoses.columns.tolist())
print("Dictionary columns:", icd_dictionary.columns.tolist())
assert not icd_dictionary.duplicated(["icd_code", "icd_version"]).any()
diagnoses_named = diagnoses.merge(
    icd_dictionary[["icd_code", "icd_version", "long_title"]],
    on=["icd_code", "icd_version"], how="left", validate="many_to_one", indicator=True
)
print(diagnoses_named._merge.value_counts())
assert len(diagnoses_named) == len(diagnoses)
display(diagnoses_named[["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version", "long_title"]].head(8))

## 8. JOIN #5 — Primary diagnosis per admission
Filter to `seq_num == 1` first. Check for duplicate `hadm_id` before merging with admissions. If duplicates appear, inspect rather than silently dropping them.

In [ ]:
primary_dx = diagnoses_named.loc[diagnoses_named.seq_num == 1, ["hadm_id", "icd_code", "icd_version", "long_title"]].copy()
print("Primary diagnosis rows:", len(primary_dx))
print("Admissions with multiple primary-diagnosis rows:", primary_dx.hadm_id.duplicated().sum())
if not primary_dx.hadm_id.duplicated().any():
    admissions_dx = admissions_a.merge(primary_dx, on="hadm_id", how="left", validate="one_to_one")
    display(admissions_dx[["hadm_id", "long_title", "los_days"]].head())
else:
    print("Review duplicated hadm_id rows before performing the join.")

## 9. JOIN #6 — Diagnosis burden per admission (aggregate then merge)
**Common pitfall:** joining raw diagnoses directly to admissions repeats an admission once per diagnosis. That inflates counts and average lengths of stay if not handled carefully. Aggregate first.

In [ ]:
dx_counts = diagnoses.groupby("hadm_id").agg(
    n_diagnosis_rows=("icd_code", "size"),
    n_distinct_icd_codes=("icd_code", "nunique")
).reset_index()
admission_dx_counts = admissions_a.merge(dx_counts, on="hadm_id", how="left", validate="one_to_one")
admission_dx_counts[["n_diagnosis_rows", "n_distinct_icd_codes"]] = admission_dx_counts[["n_diagnosis_rows", "n_distinct_icd_codes"]].fillna(0)
display(admission_dx_counts[["hadm_id", "los_days", "n_diagnosis_rows", "n_distinct_icd_codes"]].head())
print("Spearman association, diagnosis rows vs LOS:", admission_dx_counts[["n_diagnosis_rows", "los_days"]].corr(method="spearman").iloc[0,1])

## 10. Explore diagnoses and code descriptions
Top diagnoses by **unique admissions** (not raw diagnosis rows). Students can modify the search phrase for their research question.

In [ ]:
top_dx = (diagnoses_named.dropna(subset=["long_title"])
          .groupby("long_title").hadm_id.nunique().sort_values(ascending=False).head(12))
display(top_dx.rename("admissions").to_frame())
top_dx.sort_values().plot(kind="barh", figsize=(9,5), title="Most frequent coded diagnoses (unique admissions)")
plt.xlabel("Admissions"); plt.tight_layout(); plt.show()

In [ ]:
# Exploratory case-finding using text: useful for learning, not a validated clinical phenotype.
phrase = "diabetes"
selected = diagnoses_named[diagnoses_named.long_title.str.contains(phrase, case=False, na=False)]
print(f"Admissions with a diagnosis description containing '{phrase}':", selected.hadm_id.nunique())
print("Unique patients:", selected.subject_id.nunique())
display(selected[["icd_code", "icd_version", "long_title"]].drop_duplicates().head(15))

## 11. JOIN #7 — Laboratory events + lab dictionary
`labevents` is event-level data. Use `itemid` to attach test labels; use `valuenum` for numeric measurements. Units and specimen fluid matter. Avoid treating every laboratory observation as independent.

In [ ]:
labs = load_table("labevents")
lab_items = load_table("d_labitems")
print("Lab columns:", labs.columns.tolist())
print("Lab dictionary columns:", lab_items.columns.tolist())
assert lab_items.itemid.is_unique
labs_named = labs.merge(lab_items[["itemid", "label", "fluid", "category"]], on="itemid", how="left", validate="many_to_one")
assert len(labs_named) == len(labs)
display(labs_named[["subject_id", "hadm_id", "itemid", "label", "fluid", "valuenum", "valueuom"]].head())

## 12. Explore lab test frequency and missingness
Count distinct patients and admissions rather than only event rows.

In [ ]:
lab_frequency = (labs_named.groupby(["itemid", "label"], dropna=False)
                 .agg(events=("itemid", "size"), patients=("subject_id", "nunique"), admissions=("hadm_id", "nunique"))
                 .sort_values("events", ascending=False).head(15))
display(lab_frequency)
print("Missing numeric values:", labs_named.valuenum.isna().mean().round(3))

## 13. JOIN #8 — One laboratory summary per admission
Find the exact test label and specimen, check units, then aggregate per `hadm_id`. Here we illustrate **Creatinine**; inspect the labels before relying on this selection.

In [ ]:
creatinine_tests = lab_items[lab_items.label.str.fullmatch("Creatinine", case=False, na=False)]
display(creatinine_tests[["itemid", "label", "fluid", "category"]])
serum_ids = creatinine_tests.loc[creatinine_tests.fluid.eq("Blood"), "itemid"]
creatinine = labs_named[labs_named.itemid.isin(serum_ids)].copy()
print("Units:", creatinine.valueuom.value_counts(dropna=False).to_dict())
creatinine["valuenum"] = pd.to_numeric(creatinine.valuenum, errors="coerce")
creatinine_summary = (creatinine.dropna(subset=["hadm_id", "valuenum"])
    .groupby("hadm_id").agg(n_creatinine=("valuenum", "size"), mean_creatinine=("valuenum", "mean"), max_creatinine=("valuenum", "max"))
    .reset_index())
clinical_admissions = admissions_a.merge(creatinine_summary, on="hadm_id", how="left", validate="one_to_one")
assert len(clinical_admissions) == len(admissions_a)
display(clinical_admissions[["hadm_id", "los_days", "n_creatinine", "mean_creatinine", "max_creatinine"]].head())

## 14. JOIN #9 — Prescriptions + admissions (event-to-admission)
Medication orders are not proof of administration or adherence. Use this example to identify hospital admissions with recorded prescription orders. Aggregate first to preserve admission-level rows.

In [ ]:
prescriptions = load_table("prescriptions", required=False)
if prescriptions is not None:
    print("Columns:", prescriptions.columns.tolist())
    display(prescriptions.head(3))
    med_summary = prescriptions.groupby("hadm_id").agg(
        n_prescription_rows=("hadm_id", "size"),
        n_unique_drug_names=("drug", "nunique")
    ).reset_index()
    admission_medications = admissions_a.merge(med_summary, on="hadm_id", how="left", validate="one_to_one")
    display(admission_medications[["hadm_id", "los_days", "n_prescription_rows", "n_unique_drug_names"]].head())

## 15. JOIN #10 — ICU stays + admissions (one-to-many)
ICU stays have their own `stay_id`. A hospitalization may have multiple ICU stays, so merging raw ICU stays to admissions may produce multiple rows per admission. First inspect the counts.

In [ ]:
icu_stays = load_table("icustays", folder=icu, required=False)
if icu_stays is not None:
    print("ICU stay columns:", icu_stays.columns.tolist())
    display(icu_stays.head(3))
    print("ICU stays per admission:")
    display(icu_stays.groupby("hadm_id").stay_id.nunique().describe())
    icu_summary = icu_stays.groupby("hadm_id").agg(
        n_icu_stays=("stay_id", "nunique"),
        total_icu_los_days=("los", "sum")
    ).reset_index()
    admissions_icu = admissions_a.merge(icu_summary, on="hadm_id", how="left", validate="one_to_one")
    admissions_icu["n_icu_stays"] = admissions_icu.n_icu_stays.fillna(0).astype(int)
    admissions_icu["had_icu_stay"] = admissions_icu.n_icu_stays.gt(0)
    print(admissions_icu.groupby("had_icu_stay").los_days.agg(["count", "mean", "median"]).round(2))

## 16. JOIN #11 — Multi-table admission-level research dataset
Combine independently aggregated patient, diagnosis, lab and ICU features. The final result should still contain **one row per admission**. This is a strong template for student capstone projects.

In [ ]:
project_df = admissions_a.merge(
    patients[["subject_id", "gender", "anchor_age"]], on="subject_id", how="left", validate="many_to_one"
).merge(
    dx_counts, on="hadm_id", how="left", validate="one_to_one"
).merge(
    creatinine_summary, on="hadm_id", how="left", validate="one_to_one"
)
if icu_stays is not None:
    project_df = project_df.merge(icu_summary, on="hadm_id", how="left", validate="one_to_one")
assert len(project_df) == len(admissions)
assert project_df.hadm_id.is_unique
print("Final project dataset:", project_df.shape)
display(project_df.head())

## 17. Exploratory analysis: distributions, associations, and missingness
Descriptive analysis is not causal inference. Small demo samples can give unstable results, and repeated hospitalizations from the same patient are not independent.

In [ ]:
numeric = [c for c in ["anchor_age", "los_days", "n_diagnosis_rows", "n_creatinine", "mean_creatinine", "n_icu_stays"] if c in project_df]
display(project_df[numeric].describe().T.round(2))
display(project_df[numeric].isna().mean().mul(100).round(1).rename("percent_missing").to_frame())
if len(numeric) >= 2:
    display(project_df[numeric].corr(method="spearman").round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
project_df["los_days"].dropna().hist(bins=20, ax=ax)
ax.set(title="Hospital length of stay", xlabel="Days", ylabel="Admissions")
plt.tight_layout(); plt.show()

## 18. JOIN #12 — Self-join / sequence analysis: time to next admission
A **self-join-like** analysis uses multiple rows from the same admission table. Sort each patient's admissions, then use `groupby().shift()` to access the next admission. A simplistic 30-day indicator is for teaching only: it does not establish a validated readmission quality measure.

In [ ]:
sequence = admissions_a.sort_values(["subject_id", "admittime", "hadm_id"]).copy()
sequence["next_admittime"] = sequence.groupby("subject_id")["admittime"].shift(-1)
sequence["days_to_next_admission"] = (sequence.next_admittime - sequence.dischtime).dt.total_seconds() / 86400
sequence["next_within_30_days"] = sequence.days_to_next_admission.between(0, 30, inclusive="both")
display(sequence[["subject_id", "hadm_id", "dischtime", "next_admittime", "days_to_next_admission", "next_within_30_days"]].head(12))
print("Admissions followed by another within 30 days (simplified):", sequence.next_within_30_days.sum())

## 19. Optional: JOIN #13 — Transfers and admissions
Transfer records may have many rows per admission. Group by admission to calculate the number of location/transfer records, then merge into an admission-level table.

In [ ]:
transfers = load_table("transfers", required=False)
if transfers is not None:
    transfer_summary = transfers.dropna(subset=["hadm_id"]).groupby("hadm_id").agg(
        n_transfer_records=("hadm_id", "size")
    ).reset_index()
    admissions_transfers = admissions_a.merge(transfer_summary, on="hadm_id", how="left", validate="one_to_one")
    display(admissions_transfers[["hadm_id", "los_days", "n_transfer_records"]].head())

## 20. Data quality checklist: verify every join
Ask: What is my unit of analysis? Are join keys unique on the expected side? Did my row count change? Are there unmatched keys? Do missing values have a clinical or technical explanation?

In [ ]:
checks = pd.DataFrame([
    {"table": "patients", "rows": len(patients), "unique_subject_id": patients.subject_id.nunique(), "unique_hadm_id": np.nan},
    {"table": "admissions", "rows": len(admissions), "unique_subject_id": admissions.subject_id.nunique(), "unique_hadm_id": admissions.hadm_id.nunique()},
    {"table": "diagnoses", "rows": len(diagnoses), "unique_subject_id": diagnoses.subject_id.nunique(), "unique_hadm_id": diagnoses.hadm_id.nunique()},
    {"table": "labs", "rows": len(labs), "unique_subject_id": labs.subject_id.nunique(), "unique_hadm_id": labs.hadm_id.nunique()},
    {"table": "project_df", "rows": len(project_df), "unique_subject_id": project_df.subject_id.nunique(), "unique_hadm_id": project_df.hadm_id.nunique()},
])
display(checks)
assert len(project_df) == admissions.hadm_id.nunique(), "Unexpected duplicate admissions in project dataset"

## 21. Export an analysis-ready dataset
This saves to the current notebook directory. Do not publish patient-level data from restricted datasets. This demo is intended for instruction.

In [ ]:
output_path = Path.cwd() / "mimic_demo_admission_analytics.csv"
project_df.to_csv(output_path, index=False)
print("Saved:", output_path)

## 22. Student project ideas and exercises
**Choose a question and define your unit of analysis before coding.**

1. **Hospital utilization:** Compare length of stay by admission type, report counts and medians.
2. **Patient characteristics:** How does anchor age relate to number of admissions? Use a patient-level table.
3. **Diagnosis patterns:** Identify the top ICD descriptions by unique admissions; compare ICD-9 and ICD-10 coding.
4. **Diagnosis burden:** Is the number of coded diagnoses associated with length of stay? Discuss coding bias.
5. **Laboratory utilization:** Which tests are most frequently ordered? Compare event counts, patient counts, and admission counts.
6. **Creatinine and length of stay:** Summarize blood creatinine measurements by admission, check units and missingness, then explore association with LOS.
7. **Medication utilization:** Compare prescription-order counts across admissions. Explain why orders are not medication administrations.
8. **ICU utilization:** Compare admissions with and without an ICU stay; avoid interpreting the association as causal.
9. **Readmission sequences:** Calculate days to the next admission; describe why this is not a validated quality measure.
10. **Data quality audit:** Find unmatched keys, missing variables, repeated records and potential many-to-many joins.

**Required project deliverables:** research question, data dictionary, unit of analysis, join diagram, code with validation checks, descriptive table, at least one figure, limitations paragraph, and reproducible notebook.

## 23. Independent practice: fill in the blanks
Try these without looking back. Then compare the row counts before and after joining.

In [ ]:
# Exercise A: One row per patient with admission counts
# patient_counts = admissions.groupby("________")["________"].nunique().reset_index(name="n_admissions")

# Exercise B: LEFT join patient age onto admissions
# result = admissions.merge(patients[["subject_id", "anchor_age"]], on="________", how="________", validate="many_to_one")

# Exercise C: Mean LOS by admission type
# admissions_a.groupby("________")["________"].mean()

# Exercise D: Why would joining raw labevents to admissions change the number of rows?
# Write your explanation in a Markdown cell.

## 24. Reflection and interpretation
- How does the choice of **patient** vs **admission** vs **event** as the unit of analysis change the answer?
- What is the difference between `inner`, `left`, and `outer` joins?
- Why should we merge ICD code dictionaries on both code and version?
- What errors arise when one admission has dozens of lab measurements?
- What are the risks of drawing conclusions from a small, deidentified demo dataset?

**Source:** PhysioNet, *MIMIC-IV Clinical Database Demo* (use the version supplied in your course Codespace): https://physionet.org/content/mimic-iv-demo/2.2/ . See the dataset documentation for variable definitions and data-use guidance.